In [ ]:
# Shared repository paths (no model or dataset is loaded here).
from pathlib import Path as _RepositoryPath
import os as _repository_os
import sys as _repository_sys

_repository_hint = _repository_os.environ.get("TESI_REPOSITORY_ROOT")
_repository_starts = (
    [_RepositoryPath(_repository_hint).expanduser().resolve()]
    if _repository_hint else
    ([_RepositoryPath(__file__).resolve().parent] if "__file__" in globals() else [])
    + [_RepositoryPath.cwd().resolve()]
)
_repository_root = next(
    (candidate for start in _repository_starts for candidate in (start, *start.parents)
     if (candidate / "project_paths.py").is_file()), None
)
if _repository_root is None:
    raise FileNotFoundError(
        "Start Jupyter inside the repository or set TESI_REPOSITORY_ROOT to its root."
    )
if str(_repository_root) not in _repository_sys.path:
    _repository_sys.path.insert(0, str(_repository_root))
from project_paths import PROJECT_ROOT as TESI_PROJECT_ROOT
from project_paths import BASELINES_PATH as TESI_BASELINES_PATH


# EvaluationExpertWeigthAveraging — valutazione e misure

`RUN_TRAINING = False` carica il modello esportato in `Results/final_weight_averaging_model`:
nessun nuovo training, nessuna fusione dei pesi e nessuna riscrittura dei checkpoint.
Per utilizzare un'altra cartella di modello esportato, impostare `SAVED_MODEL_DIR_OVERRIDE`.
`RUN_TRAINING = True` riattiva il training originale e il successivo salvataggio.

F1 binaria sul sentiment (classe positiva 1), macro su MNLI. Per ogni task si salva
un CSV con identificativo, hash dei token, etichetta e predizione; costi e latenza
si aggiungono al JSON dei risultati esistente. Timing del forward completo su input
già tokenizzati, con warm-up e sincronizzazione CUDA, senza contatore FLOPs.
La deviazione standard della latenza è tra ripetizioni del timing, non tra seed.


In [ ]:
RUN_TRAINING = False  # Default: load the saved baseline and measure inference only.
SAVED_MODEL_DIR_OVERRIDE = None

# ============================================================
# INITIALIZATION: PATHS AND FOLDERS
# ============================================================

from pathlib import Path
import os

# ------------------------------------------------------------
# Working directory
# ------------------------------------------------------------
WORK_DIR = Path(str(TESI_PROJECT_ROOT / 'Secondo_Esperimento/Baselines/EvaluationExpertWeigthAveraging'))

# Output directories
RESULTS_DIR = WORK_DIR / "Results"
MODELS_DIR = WORK_DIR / "Models"

# ------------------------------------------------------------
# Local fine-tuned experts directory
# ------------------------------------------------------------
TRAINED_EXPERTS_DIR = Path(
    str(TESI_PROJECT_ROOT / 'Secondo_Esperimento/Baselines/TrainingOfUncasedModel/Models')
)

SST_EXPERT_DIR = TRAINED_EXPERTS_DIR / "sst_bert_base_uncased"
MNLI_EXPERT_DIR = TRAINED_EXPERTS_DIR / "mnli_bert_base_uncased"

SST_BACKBONE_DIR = SST_EXPERT_DIR / "bert_backbone"
MNLI_BACKBONE_DIR = MNLI_EXPERT_DIR / "bert_backbone"

# ------------------------------------------------------------
# Tokenizer
# Categorically use google-bert/bert-base-uncased tokenizer
# ------------------------------------------------------------
TOKENIZER_ID = "google-bert/bert-base-uncased"

# ------------------------------------------------------------
# Merged backbone output directory
# ------------------------------------------------------------
MERGED_BACKBONE_DIR = MODELS_DIR / "merged_weight_averaging_backbone"

# ------------------------------------------------------------
# Dataset directory
# ------------------------------------------------------------
DATASET_DIR = Path(str(TESI_PROJECT_ROOT / 'Secondo_Esperimento/Preprocessing/Dataset/ProcessedDB'))

SST_DIR = DATASET_DIR / "SST"
MNLI_DIR = DATASET_DIR / "MNLI"

# ------------------------------------------------------------
# Create output folders if they do not exist
# ------------------------------------------------------------
WORK_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
if RUN_TRAINING:
    MERGED_BACKBONE_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Check required directories
# ------------------------------------------------------------
required_dirs = {
    "WORK_DIR": WORK_DIR,
    "RESULTS_DIR": RESULTS_DIR,
    "MODELS_DIR": MODELS_DIR,
    "TRAINED_EXPERTS_DIR": TRAINED_EXPERTS_DIR,
    "SST_EXPERT_DIR": SST_EXPERT_DIR,
    "MNLI_EXPERT_DIR": MNLI_EXPERT_DIR,
    "SST_BACKBONE_DIR": SST_BACKBONE_DIR,
    "MNLI_BACKBONE_DIR": MNLI_BACKBONE_DIR,
    "DATASET_DIR": DATASET_DIR,
    "SST_DIR": SST_DIR,
    "MNLI_DIR": MNLI_DIR,
}

if not RUN_TRAINING:
    for key in ("TRAINED_EXPERTS_DIR", "SST_EXPERT_DIR", "MNLI_EXPERT_DIR", "SST_BACKBONE_DIR", "MNLI_BACKBONE_DIR"):
        required_dirs.pop(key, None)

for name, path in required_dirs.items():
    if not path.exists():
        raise FileNotFoundError(f"{name} does not exist: {path}")

# ------------------------------------------------------------
# Dataset files
# ------------------------------------------------------------
sst_files = {
    "X_train": SST_DIR / "X_train.parquet",
    "X_val": SST_DIR / "X_val.parquet",
    "X_test": SST_DIR / "X_test.parquet",
    "y_train": SST_DIR / "y_train.parquet",
    "y_val": SST_DIR / "y_val.parquet",
    "y_test": SST_DIR / "y_test.parquet",
}

mnli_files = {
    "X_train": MNLI_DIR / "X_train.parquet",
    "X_val": MNLI_DIR / "X_val.parquet",
    "X_test": MNLI_DIR / "X_test.parquet",
    "y_train": MNLI_DIR / "y_train.parquet",
    "y_val": MNLI_DIR / "y_val.parquet",
    "y_test": MNLI_DIR / "y_test.parquet",
}

# ------------------------------------------------------------
# Check required dataset files
# ------------------------------------------------------------
for task_name, files in {"SST": sst_files, "MNLI": mnli_files}.items():
    for split_name, file_path in files.items():
        if not file_path.exists():
            raise FileNotFoundError(f"{task_name} file missing: {split_name} -> {file_path}")

# ------------------------------------------------------------
# Set notebook working directory
# ------------------------------------------------------------
os.chdir(WORK_DIR)

print("Initialization completed successfully.")
print(f"Working directory: {WORK_DIR}")
print(f"Results directory: {RESULTS_DIR}")
print(f"Models directory: {MODELS_DIR}")

print("\nLocal experts:")
print(f"SST expert directory: {SST_EXPERT_DIR}")
print(f"MNLI expert directory: {MNLI_EXPERT_DIR}")
print(f"SST backbone directory: {SST_BACKBONE_DIR}")
print(f"MNLI backbone directory: {MNLI_BACKBONE_DIR}")

print("\nTokenizer:")
print(f"Tokenizer ID: {TOKENIZER_ID}")

print("\nDataset:")
print(f"SST directory: {SST_DIR}")
print(f"MNLI directory: {MNLI_DIR}")

print("\nMerged backbone:")
print(f"Merged backbone directory: {MERGED_BACKBONE_DIR}")

SAVED_MODEL_DIR = (Path(SAVED_MODEL_DIR_OVERRIDE) if SAVED_MODEL_DIR_OVERRIDE is not None
                   else RESULTS_DIR / "final_weight_averaging_model")
print("Mode:", "TRAINING + EVALUATION" if RUN_TRAINING else "SAVED-CHECKPOINT EVALUATION ONLY")


In [ ]:
# ============================================================
# LOAD PROCESSED DATASETS
# ============================================================

import pandas as pd

# ------------------------------------------------------------
# Helper function
# ------------------------------------------------------------
def load_parquet_split(files_dict):
    X_train = pd.read_parquet(files_dict["X_train"])
    X_val   = pd.read_parquet(files_dict["X_val"])
    X_test  = pd.read_parquet(files_dict["X_test"])

    y_train = pd.read_parquet(files_dict["y_train"])
    y_val   = pd.read_parquet(files_dict["y_val"])
    y_test  = pd.read_parquet(files_dict["y_test"])

    # Convert labels from DataFrame to Series if they have one column
    if isinstance(y_train, pd.DataFrame) and y_train.shape[1] == 1:
        y_train = y_train.iloc[:, 0]
    if isinstance(y_val, pd.DataFrame) and y_val.shape[1] == 1:
        y_val = y_val.iloc[:, 0]
    if isinstance(y_test, pd.DataFrame) and y_test.shape[1] == 1:
        y_test = y_test.iloc[:, 0]

    return X_train, X_val, X_test, y_train, y_val, y_test


# ------------------------------------------------------------
# Load SST
# ------------------------------------------------------------
X_sst_train, X_sst_val, X_sst_test, y_sst_train, y_sst_val, y_sst_test = load_parquet_split(sst_files)

# ------------------------------------------------------------
# Load MNLI
# ------------------------------------------------------------
X_mnli_train, X_mnli_val, X_mnli_test, y_mnli_train, y_mnli_val, y_mnli_test = load_parquet_split(mnli_files)


# ------------------------------------------------------------
# Basic checks
# ------------------------------------------------------------
assert len(X_sst_train) == len(y_sst_train)
assert len(X_sst_val) == len(y_sst_val)
assert len(X_sst_test) == len(y_sst_test)

assert len(X_mnli_train) == len(y_mnli_train)
assert len(X_mnli_val) == len(y_mnli_val)
assert len(X_mnli_test) == len(y_mnli_test)


# ------------------------------------------------------------
# Print dataset information
# ------------------------------------------------------------
print("SST loaded successfully")
print("X_sst_train:", X_sst_train.shape)
print("X_sst_val:  ", X_sst_val.shape)
print("X_sst_test: ", X_sst_test.shape)
print("y_sst_train:", y_sst_train.shape)
print("SST labels:", sorted(y_sst_train.unique()))

print("\nMNLI loaded successfully")
print("X_mnli_train:", X_mnli_train.shape)
print("X_mnli_val:  ", X_mnli_val.shape)
print("X_mnli_test: ", X_mnli_test.shape)
print("y_mnli_train:", y_mnli_train.shape)
print("MNLI labels:", sorted(y_mnli_train.unique()))

print("\nSST columns:", list(X_sst_train.columns))
print("MNLI columns:", list(X_mnli_train.columns))

In [ ]:
# ============================================================
# DEBUGGING TO BE REMOVED: USE ONLY A FRACTION OF THE DATA
# ============================================================

from sklearn.model_selection import train_test_split

# Modify this value when needed
DATA_FRACTION = 1
RANDOM_STATE = 42


def reduce_dataset_fraction(X, y, fraction=0.10, random_state=42):
    """
    Keep only a fraction of the dataset.
    Stratification is used to preserve label distribution.
    """
    if fraction >= 1.0:
        return X.reset_index(drop=True), y.reset_index(drop=True)

    X_small, _, y_small, _ = train_test_split(
        X,
        y,
        train_size=fraction,
        random_state=random_state,
        stratify=y
    )

    return X_small.reset_index(drop=True), y_small.reset_index(drop=True)


# ------------------------------------------------------------
# Reduce SST
# ------------------------------------------------------------
X_sst_train, y_sst_train = reduce_dataset_fraction(X_sst_train, y_sst_train, DATA_FRACTION, RANDOM_STATE)
X_sst_val,   y_sst_val   = reduce_dataset_fraction(X_sst_val,   y_sst_val,   DATA_FRACTION, RANDOM_STATE)
X_sst_test,  y_sst_test  = reduce_dataset_fraction(X_sst_test,  y_sst_test,  DATA_FRACTION, RANDOM_STATE)

# ------------------------------------------------------------
# Reduce MNLI
# ------------------------------------------------------------
X_mnli_train, y_mnli_train = reduce_dataset_fraction(X_mnli_train, y_mnli_train, DATA_FRACTION, RANDOM_STATE)
X_mnli_val,   y_mnli_val   = reduce_dataset_fraction(X_mnli_val,   y_mnli_val,   DATA_FRACTION, RANDOM_STATE)
X_mnli_test,  y_mnli_test  = reduce_dataset_fraction(X_mnli_test,  y_mnli_test,  DATA_FRACTION, RANDOM_STATE)


print(f"Using {DATA_FRACTION * 100:.1f}% of each dataset")

print("\nSST")
print("Train:", X_sst_train.shape, y_sst_train.shape)
print("Val:  ", X_sst_val.shape, y_sst_val.shape)
print("Test: ", X_sst_test.shape, y_sst_test.shape)

print("\nMNLI")
print("Train:", X_mnli_train.shape, y_mnli_train.shape)
print("Val:  ", X_mnli_val.shape, y_mnli_val.shape)
print("Test: ", X_mnli_test.shape, y_mnli_test.shape)

In [ ]:
if RUN_TRAINING:
    # ============================================================
    # LOAD LOCAL EXPERT BACKBONES AND GOOGLE-BERT TOKENIZER
    # ============================================================

    import sys
    import torch
    import numpy as np
    import torch.nn as nn
    import gc

    from transformers import AutoTokenizer, AutoModel

    # ============================================================
    # PARAMETERS
    # ============================================================
    SEED = 42

    # ============================================================
    # PATHS
    # ============================================================
    BASELINES_PATH = str(TESI_PROJECT_ROOT / 'Secondo_Esperimento/Baselines')

    # ============================================================
    # IMPORT CLASSIFIER
    # ============================================================
    if BASELINES_PATH not in sys.path:
        sys.path.append(BASELINES_PATH)

    from classifier import Classifier

    # ============================================================
    # SET RANDOM SEEDS
    # ============================================================
    torch.manual_seed(SEED)
    np.random.seed(SEED)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(SEED)
        torch.cuda.manual_seed_all(SEED)

    # ============================================================
    # DEVICE
    # ============================================================
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # ============================================================
    # TOKENIZER
    # Always use google-bert/bert-base-uncased tokenizer
    # ============================================================
    tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_ID)

    # ============================================================
    # LOAD LOCAL EXPERT BACKBONES
    # These are already saved without classification heads
    # ============================================================
    sst_backbone = AutoModel.from_pretrained(SST_BACKBONE_DIR)
    mnli_backbone = AutoModel.from_pretrained(MNLI_BACKBONE_DIR)

    # ============================================================
    # FREEZE EXPERT BACKBONES
    # They are only used to compute the averaged backbone
    # ============================================================
    sst_backbone.eval()
    mnli_backbone.eval()

    for param in sst_backbone.parameters():
        param.requires_grad = False

    for param in mnli_backbone.parameters():
        param.requires_grad = False

    # Keep them on CPU for state_dict averaging to save GPU memory
    sst_backbone = sst_backbone.cpu()
    mnli_backbone = mnli_backbone.cpu()

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print("Loaded local SST backbone from:", SST_BACKBONE_DIR)
    print("Loaded local MNLI backbone from:", MNLI_BACKBONE_DIR)
    print("Tokenizer loaded from:", TOKENIZER_ID)
    print("Classification heads are not present because AutoModel was used.")
    print("SST backbone ready.")
    print("MNLI backbone ready.")
    print("Device:", device)

    # ============================================================
    # PRINT BACKBONE ARCHITECTURES
    # ============================================================

    print("\n" + "=" * 80)
    print("SST BACKBONE ARCHITECTURE")
    print("=" * 80)
    print(sst_backbone)

    print("\n" + "=" * 80)
    print("MNLI BACKBONE ARCHITECTURE")
    print("=" * 80)
    print(mnli_backbone)

    # ============================================================
    # CHECK THAT CLASSIFIER HEADS ARE NOT PRESENT
    # ============================================================

    print("\nSST backbone type:", type(sst_backbone))
    print("MNLI backbone type:", type(mnli_backbone))

    print("\nSST has classifier:", hasattr(sst_backbone, "classifier"))
    print("MNLI has classifier:", hasattr(mnli_backbone, "classifier"))
else:

    # Load a complete saved baseline: no optimizer, weight merging or training.
    import json
    import sys
    import torch
    import numpy as np
    import torch.nn as nn
    from transformers import AutoTokenizer, AutoModel

    BASELINES_PATH = str(WORK_DIR.parent)
    if BASELINES_PATH not in sys.path:
        sys.path.append(BASELINES_PATH)
    from classifier import Classifier
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    SEED = 42
    MODEL_ID = "google-bert/bert-base-uncased"
    required = [SAVED_MODEL_DIR / "merged_backbone", SAVED_MODEL_DIR / "tokenizer",
                SAVED_MODEL_DIR / "sst_classifier.pt", SAVED_MODEL_DIR / "mnli_classifier.pt"]
    missing = [str(p) for p in required if not p.exists()]
    if missing:
        raise FileNotFoundError("Saved baseline incomplete. Point SAVED_MODEL_DIR_OVERRIDE at its exported model directory: "
                                + "; ".join(missing))
    metadata_path = SAVED_MODEL_DIR / "model_metadata.json"
    saved_metadata = json.loads(metadata_path.read_text(encoding="utf-8")) if metadata_path.is_file() else {}
    previous_path = RESULTS_DIR / "weight_averaging_test_results.json"
    previous_results = json.loads(previous_path.read_text(encoding="utf-8")) if previous_path.is_file() else {}
    training_summary_path = RESULTS_DIR / "weight_averaging_training_summary.json"
    saved_training_summary = (json.loads(training_summary_path.read_text(encoding="utf-8"))
                              if training_summary_path.is_file() else {})
    tokenizer = AutoTokenizer.from_pretrained(SAVED_MODEL_DIR / "tokenizer", local_files_only=True)
    merged_backbone = AutoModel.from_pretrained(SAVED_MODEL_DIR / "merged_backbone", local_files_only=True).to(device)
    merged_backbone.eval()
    merged_backbone.requires_grad_(False)
    sst_classifier = Classifier(class_number=2).to(device)
    mnli_classifier = Classifier(class_number=3).to(device)
    sst_classifier.load_state_dict(torch.load(SAVED_MODEL_DIR / "sst_classifier.pt", map_location=device, weights_only=True))
    mnli_classifier.load_state_dict(torch.load(SAVED_MODEL_DIR / "mnli_classifier.pt", map_location=device, weights_only=True))
    sst_classifier.eval()
    mnli_classifier.eval()
    sst_classifier.requires_grad_(False)
    mnli_classifier.requires_grad_(False)
    criterion = nn.CrossEntropyLoss()
    best_epoch = saved_metadata.get("best_epoch")
    best_val_f1_avg = saved_metadata.get("best_val_f1_avg")
    # Historical training cost is reported only when saved; this evaluation does not train.
    training_cost_report = previous_results.get("training_cost") or saved_training_summary.get("training_cost")
    sst_head_params = sum(p.numel() for p in sst_classifier.parameters())
    mnli_head_params = sum(p.numel() for p in mnli_classifier.parameters())
    bert_trainable_params = sum(p.numel() for p in merged_backbone.parameters()) if "weight_averaging" == "multitask_bert" else 0
    frozen_backbone_params = sum(p.numel() for p in merged_backbone.parameters())
    trainable_params = bert_trainable_params + sst_head_params + mnli_head_params
    ALPHA = saved_metadata.get("alpha")
    BETA = saved_metadata.get("beta")
    print("Evaluation only; loaded:", SAVED_MODEL_DIR)
    print("Inference parameters requiring gradients:", sum(
        p.numel() for module in (merged_backbone, sst_classifier, mnli_classifier)
        for p in module.parameters() if p.requires_grad))


In [ ]:
if RUN_TRAINING:
    # ============================================================
    # WEIGHT AVERAGING: MERGE LOCAL SST AND MNLI BACKBONES
    # ============================================================

    import copy
    import torch

    # ------------------------------------------------------------
    # Merge weights
    # ------------------------------------------------------------
    ALPHA = 0.5  # SST expert weight
    BETA = 0.5   # MNLI expert weight

    assert abs((ALPHA + BETA) - 1.0) < 1e-8, "ALPHA + BETA must be equal to 1."

    # ------------------------------------------------------------
    # Get backbone state dictionaries
    # ------------------------------------------------------------
    sst_state_dict = sst_backbone.state_dict()
    mnli_state_dict = mnli_backbone.state_dict()

    assert sst_state_dict.keys() == mnli_state_dict.keys(), "Backbone state_dict keys do not match."

    # ------------------------------------------------------------
    # Create averaged state dictionary
    # Formula:
    # W_M = ALPHA * W_SST + BETA * W_MNLI
    # ------------------------------------------------------------
    avg_state_dict = {}
    merge_float_elements = 0
    merge_float_tensors = 0

    for key in sst_state_dict.keys():
        sst_tensor = sst_state_dict[key]
        mnli_tensor = mnli_state_dict[key]

        assert sst_tensor.shape == mnli_tensor.shape, f"Shape mismatch for key: {key}"

        if torch.is_floating_point(sst_tensor):
            avg_state_dict[key] = ALPHA * sst_tensor + BETA * mnli_tensor
            merge_float_elements += int(sst_tensor.numel())
            merge_float_tensors += 1
        else:
            assert torch.equal(sst_tensor, mnli_tensor), f"Non-floating tensor differs for key: {key}"
            avg_state_dict[key] = sst_tensor.clone()


    # One-time model preparation, separate from training and per-example inference.
    merge_cost = {
        "method": "executed_weight_average_formula_v1",
        "scope": "One execution of ALPHA * W_SST + BETA * W_MNLI on floating state_dict tensors",
        "total_flops": int(3 * merge_float_elements),
        "num_floating_elements": int(merge_float_elements),
        "num_floating_tensors": int(merge_float_tensors),
        "alpha": float(ALPHA),
        "beta": float(BETA),
        "coverage": {
            "status": "complete_for_convention",
            "uncovered_operators": [],
            "counted_flops": int(3 * merge_float_elements),
            "counted_flops_by_operator": {
                "mul": int(2 * merge_float_elements),
                "add": int(merge_float_elements),
            },
        },
        "convention": {
            "flops_per_floating_element": 3,
            "formula": "Two scalar multiplications and one addition per merged element",
            "excluded": "Copies, non-floating buffers, comparisons, model loading and file I/O",
            "counted_once": True,
        },
    }

    # ------------------------------------------------------------
    # Create merged backbone
    # Same architecture as BERT backbone, but with averaged weights
    # ------------------------------------------------------------
    avg_backbone = copy.deepcopy(sst_backbone)
    avg_backbone.load_state_dict(avg_state_dict)

    # ------------------------------------------------------------
    # Freeze merged backbone
    # ------------------------------------------------------------
    avg_backbone.eval()

    for param in avg_backbone.parameters():
        param.requires_grad = False

    avg_backbone = avg_backbone.to(device)

    # ------------------------------------------------------------
    # Save merged frozen backbone
    # MERGED_BACKBONE_DIR was already defined in initialization
    # ------------------------------------------------------------
    MERGED_BACKBONE_DIR.mkdir(parents=True, exist_ok=True)

    avg_backbone.save_pretrained(MERGED_BACKBONE_DIR)

    # Save google-bert/bert-base-uncased tokenizer with the merged model
    tokenizer.save_pretrained(MERGED_BACKBONE_DIR)

    # ------------------------------------------------------------
    # Checks
    # ------------------------------------------------------------
    trainable_backbone_params = sum(
        p.numel() for p in avg_backbone.parameters() if p.requires_grad
    )

    total_backbone_params = sum(
        p.numel() for p in avg_backbone.parameters()
    )

    print("Weight averaging completed successfully.")
    print(f"ALPHA SST: {ALPHA}")
    print(f"BETA MNLI: {BETA}")
    print(f"Total backbone parameters: {total_backbone_params:,}")
    print(f"Trainable backbone parameters: {trainable_backbone_params:,}")
    print(f"Merged backbone saved in: {MERGED_BACKBONE_DIR}")


In [ ]:
if RUN_TRAINING:
    # ============================================================
    # CLEAN MEMORY AND LOAD ONLY THE MERGED BACKBONE
    # ============================================================

    import gc
    import torch
    from transformers import AutoModel

    # ------------------------------------------------------------
    # Remove objects that are no longer needed
    # ------------------------------------------------------------
    objects_to_delete = [
        "sst_backbone",
        "mnli_backbone",
        "sst_state_dict",
        "mnli_state_dict",
        "avg_state_dict",
        "avg_backbone",
    ]

    for obj_name in objects_to_delete:
        if obj_name in globals():
            del globals()[obj_name]

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # ------------------------------------------------------------
    # Load merged backbone from disk
    # ------------------------------------------------------------
    merged_backbone = AutoModel.from_pretrained(MERGED_BACKBONE_DIR)

    # ------------------------------------------------------------
    # Freeze merged backbone
    # ------------------------------------------------------------
    merged_backbone.eval()

    for param in merged_backbone.parameters():
        param.requires_grad = False

    merged_backbone = merged_backbone.to(device)

    # ------------------------------------------------------------
    # Checks
    # ------------------------------------------------------------
    merged_backbone_total_params = sum(
        p.numel() for p in merged_backbone.parameters()
    )

    merged_backbone_trainable_params = sum(
        p.numel() for p in merged_backbone.parameters()
        if p.requires_grad
    )

    print("Memory cleaned successfully.")
    print("Merged backbone loaded from disk.")
    print(f"Merged backbone path: {MERGED_BACKBONE_DIR}")
    print(f"Total backbone parameters: {merged_backbone_total_params:,}")
    print(f"Trainable backbone parameters: {merged_backbone_trainable_params:,}")
    print(f"Device: {device}")


In [ ]:
if RUN_TRAINING:
    # ============================================================
    # IMPORT CLASSIFIER AND CREATE TASK-SPECIFIC HEADS
    # ============================================================

    import sys

    # ------------------------------------------------------------
    # Add baselines folder to Python path
    # ------------------------------------------------------------
    BASELINES_PATH = str(TESI_PROJECT_ROOT / 'Secondo_Esperimento/Baselines')

    if BASELINES_PATH not in sys.path:
        sys.path.append(BASELINES_PATH)

    from classifier import Classifier

    # ------------------------------------------------------------
    # Create task-specific classifiers
    # ------------------------------------------------------------
    sst_classifier = Classifier(class_number=2).to(device)
    mnli_classifier = Classifier(class_number=3).to(device)

    # ------------------------------------------------------------
    # Make classifiers trainable
    # ------------------------------------------------------------
    sst_classifier.train()
    mnli_classifier.train()

    for param in sst_classifier.parameters():
        param.requires_grad = True

    for param in mnli_classifier.parameters():
        param.requires_grad = True

    # ------------------------------------------------------------
    # Count trainable parameters
    # ------------------------------------------------------------
    sst_head_params = sum(p.numel() for p in sst_classifier.parameters() if p.requires_grad)
    mnli_head_params = sum(p.numel() for p in mnli_classifier.parameters() if p.requires_grad)

    total_head_params = sst_head_params + mnli_head_params
    trainable_params = total_head_params

    print("SST classifier created.")
    print("MNLI classifier created.")
    print(f"SST classifier trainable parameters: {sst_head_params:,}")
    print(f"MNLI classifier trainable parameters: {mnli_head_params:,}")
    print(f"Total trainable head parameters: {total_head_params:,}")


In [ ]:
# ============================================================
# TOKENIZATION AND TASK-SPECIFIC DATALOADERS
# ============================================================

from tqdm import tqdm
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import torch
import pandas as pd

# ============================================================
# BATCH SIZE
# ============================================================
BATCH_SIZE = 32

# ============================================================
# Utility: convert X/y to clean lists
# ============================================================
def extract_texts(X):
    if isinstance(X, pd.DataFrame):
        if X.shape[1] != 1:
            raise ValueError(f"Expected X with 1 column, found {X.shape[1]}")
        return X.iloc[:, 0].astype(str).tolist()

    if isinstance(X, pd.Series):
        return X.astype(str).tolist()

    return pd.Series(X).astype(str).tolist()


def extract_labels(y):
    if isinstance(y, pd.DataFrame):
        if y.shape[1] != 1:
            raise ValueError(f"Expected y with 1 column, found {y.shape[1]}")
        return y.iloc[:, 0].astype(int).values

    if isinstance(y, pd.Series):
        return y.astype(int).values

    return pd.Series(y).astype(int).values


# ============================================================
# Utility: prepare inputs
# MNLI is sentence-pair classification
# SST is single-sentence classification
# ============================================================
def prepare_texts(task, X):
    texts = extract_texts(X)

    if task == "mnli":
        premises = []
        hypotheses = []

        for t in texts:
            parts = t.split("[SEP]", 1)
            premises.append(parts[0].strip())
            hypotheses.append(parts[1].strip() if len(parts) > 1 else "")

        return premises, hypotheses

    return texts, None


# ============================================================
# Inspect tokenized length distribution
# ============================================================
lengths = []

all_splits_for_length = [
    ("sst", X_sst_train),
    ("sst", X_sst_val),
    ("sst", X_sst_test),
    ("mnli", X_mnli_train),
    ("mnli", X_mnli_val),
    ("mnli", X_mnli_test),
]

for task, X_split in all_splits_for_length:
    texts_a, texts_b = prepare_texts(task, X_split)

    for i in tqdm(range(len(texts_a)), desc=f"Inspecting lengths {task}"):
        if texts_b is None:
            enc = tokenizer(texts_a[i], truncation=False)
        else:
            enc = tokenizer(texts_a[i], texts_b[i], truncation=False)

        lengths.append(len(enc["input_ids"]))

percentiles = [90, 95, 97, 99]

print("Tokenized length percentiles:")
for p in percentiles:
    print(f"{p}% :", int(np.percentile(lengths, p)))

# Match RoutedModel: fixed truncation cap, padding to the longest item in each split.
# The percentiles above are diagnostic only; they do not change this shared cap.
MAX_LEN = 256 if RUN_TRAINING else int(saved_metadata.get("max_len", 256))

print("Truncation cap (same as RoutedModel):", MAX_LEN)


# ============================================================
# Tokenization utility
# ============================================================
def tokenize_split(X, y, task, split_name):
    texts_a, texts_b = prepare_texts(task, X)
    labels_array = extract_labels(y)

    if len(texts_a) != len(labels_array):
        raise ValueError(f"Length mismatch in {split_name}: X={len(texts_a)}, y={len(labels_array)}")
    if not texts_a:
        raise ValueError(f"Cannot tokenize an empty split: {split_name}")

    # Tokenize in chunks without padding. Padding each chunk independently
    # would produce different widths and would not match RoutedModel.
    encoded_split = {"input_ids": [], "attention_mask": [], "token_type_ids": []}
    for i in tqdm(range(0, len(texts_a), 1024), desc=f"Tokenizing {split_name}"):
        batch_a = texts_a[i:i + 1024]
        batch_b = texts_b[i:i + 1024] if texts_b is not None else None
        enc = tokenizer(
            batch_a,
            text_pair=batch_b,
            padding=False,
            truncation=True,
            max_length=MAX_LEN,
            return_token_type_ids=True
        )
        encoded_split["input_ids"].extend(enc["input_ids"])
        encoded_split["attention_mask"].extend(enc["attention_mask"])
        encoded_split["token_type_ids"].extend(
            enc.get("token_type_ids", [[0] * len(ids) for ids in enc["input_ids"]])
        )

    # Pad once across the entire split, after truncation, exactly as the
    # routed tokenizer(texts, text_pair=..., padding=True, max_length=256).
    encodings = tokenizer.pad(encoded_split, padding=True, return_tensors="pt")
    input_ids = encodings["input_ids"]
    attention_mask = encodings["attention_mask"]
    token_type_ids = encodings["token_type_ids"]
    labels = torch.tensor(labels_array, dtype=torch.long)

    print(f"{split_name}: padded sequence length = {input_ids.shape[1]} (cap {MAX_LEN})")
    return TensorDataset(input_ids, attention_mask, token_type_ids, labels)


# ============================================================
# Build SST datasets
# ============================================================
sst_train_dataset = tokenize_split(X_sst_train, y_sst_train, "sst", "sst_train")
sst_val_dataset   = tokenize_split(X_sst_val,   y_sst_val,   "sst", "sst_val")
sst_test_dataset  = tokenize_split(X_sst_test,  y_sst_test,  "sst", "sst_test")

# ============================================================
# Build MNLI datasets
# ============================================================
mnli_train_dataset = tokenize_split(X_mnli_train, y_mnli_train, "mnli", "mnli_train")
mnli_val_dataset   = tokenize_split(X_mnli_val,   y_mnli_val,   "mnli", "mnli_val")
mnli_test_dataset  = tokenize_split(X_mnli_test,  y_mnli_test,  "mnli", "mnli_test")


# ============================================================
# Build task-specific dataloaders
# Each dataloader contains only one task
# ============================================================
sst_train_loader = DataLoader(sst_train_dataset, batch_size=BATCH_SIZE, shuffle=True)
sst_val_loader   = DataLoader(sst_val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
sst_test_loader  = DataLoader(sst_test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

mnli_train_loader = DataLoader(mnli_train_dataset, batch_size=BATCH_SIZE, shuffle=True)
mnli_val_loader   = DataLoader(mnli_val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
mnli_test_loader  = DataLoader(mnli_test_dataset,  batch_size=BATCH_SIZE, shuffle=False)


# ============================================================
# Loader dictionaries for training loop
# ============================================================
train_loaders = {
    "sst": sst_train_loader,
    "mnli": mnli_train_loader,
}

val_loaders = {
    "sst": sst_val_loader,
    "mnli": mnli_val_loader,
}

test_loaders = {
    "sst": sst_test_loader,
    "mnli": mnli_test_loader,
}


print("Tokenization completed successfully.")
print("MAX_LEN:", MAX_LEN)
print("SST train batches:", len(sst_train_loader))
print("MNLI train batches:", len(mnli_train_loader))

In [ ]:
if RUN_TRAINING:
    # ============================================================
    # TRAINING MECHANISMS INITIALIZATION
    # ============================================================

    import torch
    import torch.nn as nn
    from torch.optim import AdamW
    from torch.optim.lr_scheduler import ReduceLROnPlateau

    # ============================================================
    # TRAINING PARAMETERS
    # ============================================================
    NUM_EPOCHS = 10

    LEARNING_RATE = 1e-3
    WEIGHT_DECAY = 0.01

    EARLY_STOPPING_PATIENCE = 4
    EARLY_STOPPING_MIN_DELTA = 1e-4

    LR_FACTOR = 0.1
    LR_PATIENCE = 2
    LR_THRESHOLD = 1e-4
    MIN_LR = 1e-6

    # ============================================================
    # LOSS FUNCTION
    # ============================================================
    criterion = nn.CrossEntropyLoss()

    # ============================================================
    # OPTIMIZER
    # Only task-specific heads are trainable
    # ============================================================
    optimizer = AdamW(
        list(sst_classifier.parameters()) + list(mnli_classifier.parameters()),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )

    # ============================================================
    # SCHEDULER
    # Reduce LR when average validation loss stops improving
    # ============================================================
    scheduler = ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=LR_FACTOR,
        patience=LR_PATIENCE,
        threshold=LR_THRESHOLD,
        threshold_mode="abs",
        min_lr=MIN_LR
    )

    # ============================================================
    # EARLY STOPPING VARIABLES
    # ============================================================
    best_val_f1_avg = -float("inf")
    best_epoch = -1
    early_stopping_counter = 0

    # ============================================================
    # BEST MODEL PATH
    # ============================================================
    BEST_MODEL_DIR = MODELS_DIR / "best_weight_averaging_model"
    BEST_MODEL_DIR.mkdir(parents=True, exist_ok=True)

    BEST_SST_HEAD_PATH = BEST_MODEL_DIR / "sst_classifier.pt"
    BEST_MNLI_HEAD_PATH = BEST_MODEL_DIR / "mnli_classifier.pt"
    BEST_TRAINING_INFO_PATH = BEST_MODEL_DIR / "training_info.json"

    # ============================================================
    # TRAINING HISTORY
    # ============================================================
    training_history = []

    # ============================================================
    # CHECK PARAMETERS
    # ============================================================
    sst_head_params = sum(
        p.numel() for p in sst_classifier.parameters()
        if p.requires_grad
    )

    mnli_head_params = sum(
        p.numel() for p in mnli_classifier.parameters()
        if p.requires_grad
    )

    trainable_params = sst_head_params + mnli_head_params

    frozen_backbone_params = sum(
        p.numel() for p in merged_backbone.parameters()
        if not p.requires_grad
    )

    merged_backbone_trainable_params = sum(
        p.numel() for p in merged_backbone.parameters()
        if p.requires_grad
    )

    # ============================================================
    # CHECK THAT BACKBONE IS FROZEN
    # ============================================================
    assert merged_backbone_trainable_params == 0, (
        f"Merged backbone should be frozen, but has "
        f"{merged_backbone_trainable_params:,} trainable parameters."
    )

    print("Training mechanisms initialized.")
    print(f"Epochs: {NUM_EPOCHS}")
    print(f"Learning rate: {LEARNING_RATE}")
    print(f"Weight decay: {WEIGHT_DECAY}")

    print(f"SST head trainable parameters: {sst_head_params:,}")
    print(f"MNLI head trainable parameters: {mnli_head_params:,}")
    print(f"Total trainable head parameters: {trainable_params:,}")

    print(f"Frozen backbone parameters: {frozen_backbone_params:,}")
    print(f"Merged backbone trainable parameters: {merged_backbone_trainable_params:,}")

    print(f"Early stopping patience: {EARLY_STOPPING_PATIENCE}")
    print(f"Early stopping min delta: {EARLY_STOPPING_MIN_DELTA}")
    print(f"LR factor: {LR_FACTOR}")
    print(f"LR patience: {LR_PATIENCE}")
    print(f"LR threshold: {LR_THRESHOLD}")
    print(f"Minimum LR: {MIN_LR}")

    print(f"Best model directory: {BEST_MODEL_DIR}")

    # ============================================================
    # COMPUTATIONAL COST: INFERENCE AND TRAINING
    # ============================================================

from math import prod
from collections import defaultdict
from torch.utils.flop_counter import FlopCounterMode
from torch.utils._pytree import tree_leaves
from transformers import __version__ as transformers_version


class InferenceFlopCounter(FlopCounterMode):
    """Extend PyTorch's matrix counts and expose uncovered floating-point ops."""

    # Indexing, copies, shape changes, allocation and comparisons have no
    # floating-point arithmetic under this convention. Dropout is off in eval.
    zero_flop_ops = set("""
        alias detach view _unsafe_view reshape t transpose permute slice select
        squeeze unsqueeze expand expand_as as_strided contiguous clone copy_
        _to_copy to empty empty_like empty_strided new_empty new_empty_strided
        zeros zeros_like new_zeros ones ones_like full full_like new_full fill_
        zero_ embedding index index_select gather cat stack split split_with_sizes
        unbind masked_fill masked_fill_ where eq ne lt le gt ge argmax argmin
        max min amax amin any all isfinite isnan isinf scalar_tensor
        _local_scalar_dense lift_fresh lift_fresh_copy resolve_conj dropout
        native_dropout constant_pad_nd
    """.split())

    def __init__(self):
        if not hasattr(FlopCounterMode, "_count_flops"):
            raise RuntimeError("This FLOPs counter requires PyTorch with FlopCounterMode._count_flops support.")
        super().__init__(display=False)
        self.uncovered_ops = set()

    def _count_flops(self, packet, out, args, kwargs):
        name = str(packet).split(".")[-1]
        outputs = [x for x in tree_leaves(out)
                   if isinstance(x, torch.Tensor) and x.is_floating_point()]
        if not outputs:
            return out
        n = outputs[0].numel()
        x = args[0] if args else None

        if name in {"addmm", "baddbmm"}:
            # PyTorch's default matrix formula omits the bias addition.
            alpha, beta = kwargs.get("alpha", 1), kwargs.get("beta", 1)
            flops = 2 * n * args[1].shape[-1]
            flops += n * (int(alpha != 1) + int(beta != 0) + int(beta not in (0, 1)))
        elif name in {"add", "add_", "sub", "sub_", "rsub"}:
            flops = n * (1 + int(kwargs.get("alpha", 1) != 1))
        elif name in {"mul", "mul_", "div", "div_", "true_divide", "neg",
                      "exp", "log", "sqrt", "rsqrt", "reciprocal", "tanh", "erf", "pow"}:
            flops = n
        elif name == "gelu":
            flops = n * (9 if kwargs.get("approximate", "none") == "tanh" else 5)
        elif name in {"native_layer_norm", "layer_norm"}:
            width = prod(args[1])
            rows = x.numel() // width
            weight = args[2] if len(args) > 2 else kwargs.get("weight")
            bias = args[3] if len(args) > 3 else kwargs.get("bias")
            flops = 5 * x.numel() + 2 * rows
            flops += x.numel() * (int(weight is not None) + int(bias is not None))
        elif name in {"_softmax", "softmax"}:
            dim = args[1] if len(args) > 1 else kwargs["dim"]
            flops = 4 * x.numel() - x.numel() // x.shape[dim]
        elif name in {"sum", "mean"}:
            flops = x.numel() - n + (n if name == "mean" else 0)
        elif name in {
            "scaled_dot_product_attention", "_scaled_dot_product_attention_math",
            "_scaled_dot_product_flash_attention", "_scaled_dot_product_flash_attention_for_cpu",
            "_scaled_dot_product_efficient_attention", "_scaled_dot_product_cudnn_attention"
        }:
            # Dense, non-causal BERT attention: QK + scaling/mask/softmax + AV.
            # Applies to the operator as a whole; no extra forward is performed.
            q, k, v = args[:3]
            rows = prod(q.shape[:-1])
            scores = rows * k.shape[-2]
            mask = kwargs.get("attn_mask", kwargs.get("attn_bias"))
            if mask is None and len(args) > 3 and isinstance(args[3], torch.Tensor):
                mask = args[3]
            flops = 2 * scores * (q.shape[-1] + v.shape[-1])
            flops += 5 * scores - rows
            if isinstance(mask, torch.Tensor) and mask.is_floating_point():
                flops += scores
        elif name in self.zero_flop_ops:
            return out
        elif packet in self.flop_registry:
            # Other fused attention variants may count only their matmuls.
            if "attention" in name:
                self.uncovered_ops.add(str(packet) + " (attention overhead not covered)")
            return super()._count_flops(packet, out, args, kwargs)
        else:
            self.uncovered_ops.add(str(packet))
            return out

        self.flop_counts["Global"][packet] += int(flops)
        return out


def new_inference_cost():
    return {
        "components": defaultdict(int), "operators": defaultdict(int),
        "uncovered": set(), "num_examples": 0,
        "sequence_lengths": set(), "batch_sizes": set(),
    }


def add_inference_cost(cost, counter, component, input_ids=None):
    cost["components"][component] += int(counter.get_total_flops())
    for op, value in counter.get_flop_counts().get("Global", {}).items():
        cost["operators"][str(op)] += int(value)
    cost["uncovered"].update(counter.uncovered_ops)
    # Count examples only on the backbone pass, not again on cached features.
    if input_ids is not None:
        cost["num_examples"] += int(input_ids.shape[0])
        cost["sequence_lengths"].add(int(input_ids.shape[1]))
        cost["batch_sizes"].add(int(input_ids.shape[0]))


def finish_inference_cost(cost, num_samples):
    if num_samples <= 0 or cost["num_examples"] != num_samples:
        raise ValueError("FLOPs: backbone and classifier must cover the same test examples.")
    counted = int(sum(cost["components"].values()))
    complete = not cost["uncovered"] and counted > 0
    if not complete:
        print("Inference FLOPs incomplete; see inference_cost.coverage in the saved results.")
    return {
        "method": "pytorch_executed_forward_flops_v1",
        "scope": "test inference: backbone and classifier, inputs to logits",
        "total_flops": counted if complete else None,
        "flops_per_example": float(counted / num_samples) if complete else None,
        "num_examples": int(num_samples),
        "coverage": {
            "status": "complete_for_convention" if complete else "incomplete",
            "uncovered_operators": sorted(cost["uncovered"]),
            "counted_flops": counted,
            "counted_flops_by_component": dict(cost["components"]),
            "counted_flops_by_operator": dict(sorted(cost["operators"].items())),
        },
        "convention": {
            "multiply_accumulate_flops": 2,
            "elementwise_arithmetic_or_special_function_flops": 1,
            "gelu_flops_per_element": {"exact": 5, "tanh_approximation": 9},
            "layer_norm": "5*N + 2*rows, plus N each for affine weight and bias",
            "softmax": "4*N - rows; comparisons excluded",
            "attention": "dense QK and AV matmuls plus scaling, additive mask and softmax",
            "excluded": "training/backward, loss/metrics, preprocessing, I/O, copies, indexing, comparisons",
            "interpretation": "formula-based FLOPs of executed operators; not exact GPU kernel instructions",
        },
        "input": {
            "max_len": int(MAX_LEN), "batch_size": int(BATCH_SIZE),
            "observed_sequence_lengths": sorted(cost["sequence_lengths"]),
            "observed_batch_sizes": sorted(cost["batch_sizes"]),
            "padding": "longest_in_split", "truncation": True,
            "tokenizer": str(TOKENIZER_ID),
            "dtype": str(next(merged_backbone.parameters()).dtype),
        },
        "torch_version": str(torch.__version__),
        "transformers_version": transformers_version,
    }



# ============================================================
# TRAINING FLOPS: extend the same convention to the actual train step
# ============================================================
class TrainingFlopCounter(InferenceFlopCounter):
    """Count training tensor arithmetic without another forward/backward pass."""

    def _count_flops(self, packet, out, args, kwargs):
        name = str(packet).split(".")[-1]
        outputs = [x for x in tree_leaves(out)
                   if isinstance(x, torch.Tensor) and x.is_floating_point()]
        n = outputs[0].numel() if outputs else 0

        # RNG generation is excluded; applying/scaling the dropout mask is counted.
        if name in {"bernoulli", "bernoulli_", "uniform_", "normal_"}:
            return out
        if name in {"dropout", "native_dropout"}:
            probability = args[1] if len(args) > 1 else kwargs["p"]
            training = args[2] if len(args) > 2 else kwargs.get("train", True)
            flops = 2 * args[0].numel() if training and 0 < probability < 1 else 0
        elif name == "native_dropout_backward":
            flops = 2 * args[0].numel()
        elif name == "gelu_backward":
            # Exact: g * (Phi(x) + x * phi(x)); scalar constants precomputed.
            approximation = kwargs.get("approximate", "none")
            flops = args[0].numel() * (18 if approximation == "tanh" else 11)
        elif name == "native_layer_norm_backward":
            # Reuse saved mean/rstd. Count only requested dx, dweight and dbias.
            x, shape, weight, mask = args[1], args[2], args[5], args[7]
            elements = x.numel()
            width = prod(shape)
            rows = elements // width
            flops = 2 * elements if mask[0] or mask[1] else 0  # xhat
            if mask[0]:
                # g=dy*weight; two row means; dx=rstd*(g-mean(g)-xhat*mean(g*xhat)).
                flops += (7 + int(weight is not None)) * elements
            if mask[1]:
                flops += elements + (elements - width)
            if mask[2]:
                flops += elements - width
        elif name in {"_log_softmax", "log_softmax"}:
            flops = 4 * args[0].numel()
        elif name == "_log_softmax_backward_data":
            grad, dim = args[0], args[2]
            flops = 4 * grad.numel() - grad.numel() // grad.shape[dim]
        elif name == "_softmax_backward_data":
            grad, dim = args[0], args[2]
            flops = 4 * grad.numel() - grad.numel() // grad.shape[dim]
        elif name in {"nll_loss_forward", "nll_loss_backward"}:
            offset = int(name == "nll_loss_backward")
            weight = args[2 + offset] if len(args) > 2 + offset else kwargs.get("weight")
            reduction = args[3 + offset] if len(args) > 3 + offset else kwargs.get("reduction", 1)
            targets = args[1 + offset].numel()
            if weight is not None:
                self.uncovered_ops.add(str(packet) + " (weighted NLL not covered)")
                return out
            # Current dataset has valid class IDs, no ignored targets.
            if offset:
                flops = targets * (1 + int(reduction == 1))
            else:
                flops = targets
                if reduction != 0:
                    flops += max(targets - 1, 0) + int(reduction == 1)
        elif name.startswith("_foreach_"):
            op = name[len("_foreach_"):].rstrip("_")
            tensors = outputs or [x for x in tree_leaves(args[0])
                                  if isinstance(x, torch.Tensor) and x.is_floating_point()]
            elements = sum(x.numel() for x in tensors)
            if op in {"mul", "div", "sqrt", "rsqrt", "reciprocal", "neg", "pow"}:
                flops = elements
            elif op in {"add", "sub"}:
                flops = elements * (1 + int(kwargs.get("alpha", 1) != 1))
            elif op == "lerp":
                flops = 3 * elements
            elif op in {"addcmul", "addcdiv"}:
                value = args[3] if len(args) > 3 else kwargs.get("value", 1)
                if isinstance(value, (list, tuple)):
                    flops = sum(t.numel() * (2 + int(v != 1)) for t, v in zip(tensors, value))
                elif isinstance(value, torch.Tensor):
                    # Tensor scale values cannot be read without a device synchronization.
                    self.uncovered_ops.add(str(packet) + " (tensor scale not covered)")
                    return out
                else:
                    flops = elements * (2 + int(value != 1))
            elif op in {"copy", "zero", "maximum", "minimum"}:
                return out
            else:
                self.uncovered_ops.add(str(packet))
                return out
        elif name in {"lerp", "lerp_"}:
            flops = 3 * n
        elif name in {"addcmul", "addcmul_", "addcdiv", "addcdiv_"}:
            flops = n * (2 + int(kwargs.get("value", 1) != 1))
        elif name in {"sqrt_", "rsqrt_", "reciprocal_", "neg_", "pow_"}:
            flops = n
        else:
            # Also flag unknown mutating operators whose return value is empty.
            if not outputs and name not in self.zero_flop_ops:
                if any(isinstance(x, torch.Tensor) and x.is_floating_point()
                       for x in tree_leaves(args)):
                    self.uncovered_ops.add(str(packet))
                return out
            return super()._count_flops(packet, out, args, kwargs)

        self.flop_counts["Global"][packet] += int(flops)
        return out


def new_training_cost():
    return {}


def add_training_cost(cost, counter, component, num_examples, input_ids=None):
    item = cost.setdefault(component, {
        "counted_flops": 0, "calls": 0, "num_examples": 0,
        "operators": defaultdict(int), "uncovered": set(),
        "sequence_lengths": set(), "batch_sizes": set(),
    })
    item["counted_flops"] += int(counter.get_total_flops())
    item["calls"] += 1
    item["num_examples"] += int(num_examples)
    item["batch_sizes"].add(int(num_examples))
    item["uncovered"].update(counter.uncovered_ops)
    for op, value in counter.get_flop_counts().get("Global", {}).items():
        item["operators"][str(op)] += int(value)
    if input_ids is not None:
        item["sequence_lengths"].add(int(input_ids.shape[1]))


def merge_training_cost(total, addition):
    for name, item in addition.items():
        if name not in total:
            total[name] = {
                "counted_flops": 0, "calls": 0, "num_examples": 0,
                "operators": defaultdict(int), "uncovered": set(),
                "sequence_lengths": set(), "batch_sizes": set(),
            }
        target = total[name]
        for key in ("counted_flops", "calls", "num_examples"):
            target[key] += item[key]
        for key in ("uncovered", "sequence_lengths", "batch_sizes"):
            target[key].update(item[key])
        for op, value in item["operators"].items():
            target["operators"][op] += value


def finish_training_cost(cost):
    components = {}
    for name, item in cost.items():
        components[name] = {
            "total_flops": item["counted_flops"] if not item["uncovered"] else None,
            "counted_flops": item["counted_flops"],
            "calls": item["calls"],
            "num_examples": item["num_examples"],
            "counted_flops_by_operator": dict(sorted(item["operators"].items())),
            "uncovered_operators": sorted(item["uncovered"]),
            "observed_sequence_lengths": sorted(item["sequence_lengths"]),
            "observed_batch_sizes": sorted(item["batch_sizes"]),
        }
    training = [v for k, v in components.items() if "validation" not in k]
    validation = [v for k, v in components.items() if "validation" in k]

    def subtotal(items):
        if any(item["total_flops"] is None for item in items):
            return None
        return sum(item["total_flops"] for item in items)

    uncovered = sorted({op for item in components.values() for op in item["uncovered_operators"]})
    return {
        "training_flops": subtotal(training),
        "validation_flops": subtotal(validation),
        "total_flops": subtotal(list(components.values())),
        "components": components,
        "coverage": {
            "status": "incomplete" if uncovered else "complete_for_convention",
            "uncovered_operators": uncovered,
            "counted_flops": sum(item["counted_flops"] for item in components.values()),
        },
    }



def finish_multitask_training_cost(cost_by_task):
    combined = new_training_cost()
    for cost in cost_by_task.values():
        merge_training_cost(combined, cost)
    report = finish_training_cost(combined)
    report["by_task"] = {task: finish_training_cost(cost) for task, cost in cost_by_task.items()}
    return report



In [ ]:
if RUN_TRAINING:
    # ============================================================
    # TRAINING LOOP: WEIGHT AVERAGING BASELINE
    # Frozen backbone caching: since the backbone is frozen, we compute
    # pooled_output once and then train only the task-specific heads.
    # ============================================================

    import json
    import random
    import torch
    from tqdm import tqdm
    from torch.utils.data import TensorDataset, DataLoader
    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

    # ============================================================
    # OUTPUT FILES
    # ============================================================

    HISTORY_PATH = RESULTS_DIR / "weight_averaging_training_history.json"
    FINAL_SUMMARY_PATH = RESULTS_DIR / "weight_averaging_training_summary.json"

    # ============================================================
    # METRICS
    # ============================================================

    def compute_metrics(y_true, y_pred, task):
        average = "binary" if task == "sst" else "macro"
        return {
            "accuracy": float(accuracy_score(y_true, y_pred)),
            "precision": float(precision_score(y_true, y_pred, average=average, zero_division=0)),
            "recall": float(recall_score(y_true, y_pred, average=average, zero_division=0)),
            "f1": float(f1_score(y_true, y_pred, average=average, zero_division=0)),
        }


    def average_metrics(metrics_sst, metrics_mnli):
        return {
            metric: float((metrics_sst[metric] + metrics_mnli[metric]) / 2)
            for metric in ["accuracy", "precision", "recall", "f1"]
        }


    # ============================================================
    # PRECOMPUTE BACKBONE FEATURES
    # ============================================================

    def cache_backbone_features(dataloader, task_name, cost, component):
        merged_backbone.eval()

        features = []
        labels_list = []

        with torch.no_grad():
            for batch in tqdm(dataloader, desc=f"Caching {task_name} features"):
                input_ids, attention_mask, token_type_ids, labels = batch

                input_ids = input_ids.to(device)
                attention_mask = attention_mask.to(device)
                token_type_ids = token_type_ids.to(device)

                with InferenceFlopCounter() as counter:
                    outputs = merged_backbone(
                        input_ids=input_ids,
                        attention_mask=attention_mask,
                        token_type_ids=token_type_ids
                    )
                add_training_cost(cost, counter, component, labels.size(0), input_ids)

                pooled_output = outputs.pooler_output.detach().cpu()

                features.append(pooled_output)
                labels_list.append(labels.detach().cpu())

        features = torch.cat(features, dim=0)
        labels = torch.cat(labels_list, dim=0)

        return TensorDataset(features, labels)


    training_cost_total = {"sst": new_training_cost(), "mnli": new_training_cost()}

    sst_train_features = cache_backbone_features(
        sst_train_loader, "SST train", training_cost_total["sst"], "backbone_cache_train"
    )
    sst_val_features = cache_backbone_features(
        sst_val_loader, "SST val", training_cost_total["sst"], "backbone_cache_validation"
    )
    mnli_train_features = cache_backbone_features(
        mnli_train_loader, "MNLI train", training_cost_total["mnli"], "backbone_cache_train"
    )
    mnli_val_features = cache_backbone_features(
        mnli_val_loader, "MNLI val", training_cost_total["mnli"], "backbone_cache_validation"
    )

    sst_train_feature_loader = DataLoader(sst_train_features, batch_size=BATCH_SIZE, shuffle=True)
    sst_val_feature_loader   = DataLoader(sst_val_features,   batch_size=BATCH_SIZE, shuffle=False)

    mnli_train_feature_loader = DataLoader(mnli_train_features, batch_size=BATCH_SIZE, shuffle=True)
    mnli_val_feature_loader   = DataLoader(mnli_val_features,   batch_size=BATCH_SIZE, shuffle=False)

    print("Backbone features cached successfully.")


    # ============================================================
    # FORWARD PASS ON CACHED FEATURES
    # ============================================================

    def forward_cached_batch(task, batch):
        pooled_output, labels = batch

        pooled_output = pooled_output.to(device)
        labels = labels.to(device)

        if task == "sst":
            logits = sst_classifier(pooled_output)
        elif task == "mnli":
            logits = mnli_classifier(pooled_output)
        else:
            raise ValueError(f"Unknown task: {task}")

        loss = criterion(logits, labels)
        preds = torch.argmax(logits, dim=1)

        return loss, preds, labels


    # ============================================================
    # TRAIN ONE EPOCH
    # ============================================================

    def train_one_epoch(epoch, cost_by_task):
        merged_backbone.eval()
        sst_classifier.train()
        mnli_classifier.train()

        num_sst_batches = len(sst_train_feature_loader)
        num_mnli_batches = len(mnli_train_feature_loader)

        assert num_sst_batches == num_mnli_batches, (
            f"Different number of batches: SST={num_sst_batches}, MNLI={num_mnli_batches}"
        )

        task_schedule = ["sst"] * num_sst_batches + ["mnli"] * num_mnli_batches
        random.shuffle(task_schedule)

        sst_iter = iter(sst_train_feature_loader)
        mnli_iter = iter(mnli_train_feature_loader)

        stats = {
            "sst": {"loss_sum": 0.0, "total": 0, "y_true": [], "y_pred": []},
            "mnli": {"loss_sum": 0.0, "total": 0, "y_true": [], "y_pred": []},
        }

        progress_bar = tqdm(
            task_schedule,
            desc=f"Epoch {epoch + 1}/{NUM_EPOCHS}",
            leave=True
        )

        for task in progress_bar:
            batch = next(sst_iter) if task == "sst" else next(mnli_iter)

            optimizer.zero_grad(set_to_none=True)

            with TrainingFlopCounter() as forward_counter:
                loss, preds, labels = forward_cached_batch(task, batch)

            with TrainingFlopCounter() as backward_counter:
                loss.backward()
            with TrainingFlopCounter() as optimizer_counter:
                optimizer.step()

            batch_size = labels.size(0)
            add_training_cost(cost_by_task[task], forward_counter, "classifier_forward_and_loss", batch_size)
            add_training_cost(cost_by_task[task], backward_counter, "backward", batch_size)
            add_training_cost(cost_by_task[task], optimizer_counter, "optimizer_step", batch_size)

            stats[task]["loss_sum"] += loss.item() * batch_size
            stats[task]["total"] += batch_size
            stats[task]["y_true"].extend(labels.detach().cpu().numpy())
            stats[task]["y_pred"].extend(preds.detach().cpu().numpy())

            progress_bar.set_postfix({
                "task": task,
                "loss": f"{loss.item():.4f}",
                "lr": f"{optimizer.param_groups[0]['lr']:.1e}"
            })

        metrics_sst = compute_metrics(stats["sst"]["y_true"], stats["sst"]["y_pred"], "sst")
        metrics_mnli = compute_metrics(stats["mnli"]["y_true"], stats["mnli"]["y_pred"], "mnli")
        metrics_avg = average_metrics(metrics_sst, metrics_mnli)

        loss_sst = float(stats["sst"]["loss_sum"] / stats["sst"]["total"])
        loss_mnli = float(stats["mnli"]["loss_sum"] / stats["mnli"]["total"])

        return {
            "loss_sst": loss_sst,
            "loss_mnli": loss_mnli,
            "loss_avg": float((loss_sst + loss_mnli) / 2),
            "metrics_sst": metrics_sst,
            "metrics_mnli": metrics_mnli,
            "metrics_avg": metrics_avg,
        }


    # ============================================================
    # EVALUATE ONE TASK
    # ============================================================

    def evaluate_task(task, dataloader, cost):
        sst_classifier.eval()
        mnli_classifier.eval()

        loss_sum = 0.0
        total = 0
        y_true = []
        y_pred = []

        with torch.no_grad():
            for batch in dataloader:
                with TrainingFlopCounter() as counter:
                    loss, preds, labels = forward_cached_batch(task, batch)

                batch_size = labels.size(0)
                add_training_cost(cost, counter, "validation_forward_and_loss", batch_size)

                loss_sum += loss.item() * batch_size
                total += batch_size

                y_true.extend(labels.detach().cpu().numpy())
                y_pred.extend(preds.detach().cpu().numpy())

        return {
            "loss": float(loss_sum / total),
            "metrics": compute_metrics(y_true, y_pred, task),
        }


    # ============================================================
    # MAIN TRAINING LOOP
    # ============================================================

    training_history = []

    best_val_f1_avg = -float("inf")
    best_epoch = -1
    early_stopping_counter = 0

    for epoch in range(NUM_EPOCHS):
        epoch_cost = {"sst": new_training_cost(), "mnli": new_training_cost()}

        print("\n" + "=" * 80)
        print(f"Epoch {epoch + 1}/{NUM_EPOCHS}")
        print("=" * 80)

        train_results = train_one_epoch(epoch, epoch_cost)

        val_sst_results = evaluate_task("sst", sst_val_feature_loader, epoch_cost["sst"])
        val_mnli_results = evaluate_task("mnli", mnli_val_feature_loader, epoch_cost["mnli"])
        for task in ("sst", "mnli"):
            merge_training_cost(training_cost_total[task], epoch_cost[task])

        val_metrics_sst = val_sst_results["metrics"]
        val_metrics_mnli = val_mnli_results["metrics"]
        val_metrics_avg = average_metrics(val_metrics_sst, val_metrics_mnli)

        val_loss_sst = val_sst_results["loss"]
        val_loss_mnli = val_mnli_results["loss"]
        val_loss_avg = float((val_loss_sst + val_loss_mnli) / 2)

        scheduler.step(val_loss_avg)

        current_lr = float(optimizer.param_groups[0]["lr"])
        val_f1_avg = val_metrics_avg["f1"]

        is_best = val_f1_avg > best_val_f1_avg + EARLY_STOPPING_MIN_DELTA

        if is_best:
            best_val_f1_avg = val_f1_avg
            best_epoch = epoch + 1
            early_stopping_counter = 0

            torch.save(sst_classifier.state_dict(), BEST_SST_HEAD_PATH)
            torch.save(mnli_classifier.state_dict(), BEST_MNLI_HEAD_PATH)

        else:
            early_stopping_counter += 1

        epoch_record = {
            "epoch": epoch + 1,

            "train_loss_sst": train_results["loss_sst"],
            "train_loss_mnli": train_results["loss_mnli"],
            "train_loss_avg": train_results["loss_avg"],

            "train_sst": train_results["metrics_sst"],
            "train_mnli": train_results["metrics_mnli"],
            "train_avg": train_results["metrics_avg"],

            "val_loss_sst": val_loss_sst,
            "val_loss_mnli": val_loss_mnli,
            "val_loss_avg": val_loss_avg,

            "val_sst": val_metrics_sst,
            "val_mnli": val_metrics_mnli,
            "val_avg": val_metrics_avg,

            "learning_rate": current_lr,
            "is_best": is_best,
            "early_stopping_counter": early_stopping_counter,
            "training_cost": finish_multitask_training_cost(epoch_cost),
        }

        training_history.append(epoch_record)

        with open(HISTORY_PATH, "w") as f:
            json.dump(training_history, f, indent=4)

        print(
            f"train_loss_avg={train_results['loss_avg']:.4f} | "
            f"val_loss_avg={val_loss_avg:.4f} | "
            f"val_f1_avg={val_f1_avg:.4f} | "
            f"val_acc_avg={val_metrics_avg['accuracy']:.4f} | "
            f"lr={current_lr:.8f} | "
            f"best={is_best}"
        )

        if early_stopping_counter >= EARLY_STOPPING_PATIENCE:
            print("\nEarly stopping triggered.")
            break


    # ============================================================
    # SAVE FINAL SUMMARY
    # ============================================================

    best_record = training_history[best_epoch - 1] if best_epoch > 0 else None
    last_record = training_history[-1]

    training_cost_report = finish_multitask_training_cost(training_cost_total)
    training_cost_report.update({
        "method": "pytorch_executed_training_flops_v1",
        "scope": "Head training with a frozen merged backbone; one-time train/validation caches included",
        "preparation_cost": merge_cost,
        "total_with_preparation_flops": (
            training_cost_report["total_flops"] + merge_cost["total_flops"]
            if training_cost_report["total_flops"] is not None else None
        ),
        "epochs_completed": int(len(training_history)),
        "backbone_caching": True,
        "backbone_frozen": True,
        "max_len": int(MAX_LEN),
        "batch_size": int(BATCH_SIZE),
        "dtype": str(next(merged_backbone.parameters()).dtype),
        "torch_version": str(torch.__version__),
        "transformers_version": transformers_version,
        "optimizer_steps": {task: cost["optimizer_step"]["calls"] for task, cost in training_cost_total.items()},
        "training_example_visits": {task: cost["classifier_forward_and_loss"]["num_examples"] for task, cost in training_cost_total.items()},
        "convention": {
            "multiply_accumulate_flops": 2,
            "elementwise_arithmetic_or_special_function_flops": 1,
            "forward": "Same backbone/classifier convention as inference_cost",
            "gelu_backward_flops_per_element": {"exact": 11, "tanh_approximation": 18},
            "layer_norm_backward": "Saved mean/rstd; xhat once; only requested input/weight/bias gradients",
            "dropout": "Mask multiplication and scaling counted; RNG generation excluded",
            "cross_entropy": "Log-softmax and unweighted NLL, forward/backward; valid class IDs, no ignored targets",
            "optimizer": "Executed AdamW tensor operations; only the active head has gradients",
            "excluded": "Tokenization, I/O, copies, indexing, comparisons, RNG, metric calculation and Python scalar bookkeeping",
            "interpretation": "Formula-based FLOPs of executed tensor operators; not exact GPU kernel instructions",
            "total": "training_flops + validation_flops; sums both tasks, one-time caches and all completed epochs",
            "total_with_preparation": "total_flops + one-time weight averaging; test inference remains separate",
            "epoch_totals": "Exclude backbone caches and weight averaging, which are counted once for the run",
        },
        "per_epoch": [
            {
                "epoch": record["epoch"],
                **{key: record["training_cost"][key] for key in ("training_flops", "validation_flops", "total_flops")},
                "by_task": {
                    task: {key: record["training_cost"]["by_task"][task][key]
                           for key in ("training_flops", "validation_flops", "total_flops")}
                    for task in ("sst", "mnli")
                },
            }
            for record in training_history
        ],
    })

    final_summary = {
        "baseline": "weight_averaging",
        "tokenizer_id": TOKENIZER_ID,

        "sst_expert_dir": str(SST_EXPERT_DIR),
        "mnli_expert_dir": str(MNLI_EXPERT_DIR),
        "sst_backbone_dir": str(SST_BACKBONE_DIR),
        "mnli_backbone_dir": str(MNLI_BACKBONE_DIR),
        "merged_backbone_dir": str(MERGED_BACKBONE_DIR),

        "best_epoch": int(best_epoch),
        "best_val_f1_avg": float(best_val_f1_avg),
        "best_record": best_record,

        "last_epoch": int(last_record["epoch"]),
        "last_record": last_record,

        "alpha": float(ALPHA),
        "beta": float(BETA),

        "max_len": int(MAX_LEN),
        "batch_size": int(BATCH_SIZE),
        "num_epochs_requested": int(NUM_EPOCHS),
        "num_epochs_completed": int(len(training_history)),

        "learning_rate": float(LEARNING_RATE),
        "weight_decay": float(WEIGHT_DECAY),

        "early_stopping_patience": int(EARLY_STOPPING_PATIENCE),
        "early_stopping_min_delta": float(EARLY_STOPPING_MIN_DELTA),

        "lr_factor": float(LR_FACTOR),
        "lr_patience": int(LR_PATIENCE),
        "lr_threshold": float(LR_THRESHOLD),
        "min_lr": float(MIN_LR),

        "sst_head_params": int(sst_head_params),
        "mnli_head_params": int(mnli_head_params),
        "trainable_params": int(trainable_params),
        "frozen_backbone_params": int(frozen_backbone_params),

        "backbone_caching": True,
        "backbone_frozen": True,
        "training_cost": training_cost_report,

        "best_model_dir": str(BEST_MODEL_DIR),
    }

    with open(FINAL_SUMMARY_PATH, "w") as f:
        json.dump(final_summary, f, indent=4)

    with open(BEST_TRAINING_INFO_PATH, "w") as f:
        json.dump(final_summary, f, indent=4)


    # ============================================================
    # FINAL PRINT ONLY
    # ============================================================

    print("\nTraining completed.")
    print(f"Epochs completed: {len(training_history)}")
    print(f"Best epoch: {best_epoch}")
    print(f"Best validation AVG F1: {best_val_f1_avg:.4f}")

    if best_record is not None:
        print(f"Best validation AVG accuracy: {best_record['val_avg']['accuracy']:.4f}")
        print(f"Best validation AVG loss: {best_record['val_loss_avg']:.4f}")

    print(f"History saved in: {HISTORY_PATH}")
    print(f"Summary saved in: {FINAL_SUMMARY_PATH}")
    print(f"Best model saved in: {BEST_MODEL_DIR}")


In [ ]:
if RUN_TRAINING:
    # ============================================================
    # SAVE FINAL WEIGHT AVERAGING MODEL IN RESULTS
    # ============================================================

    import json
    import torch

    # ------------------------------------------------------------
    # Output directory inside Results
    # ------------------------------------------------------------
    FINAL_MODEL_RESULTS_DIR = RESULTS_DIR / "final_weight_averaging_model"
    FINAL_MODEL_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

    # ------------------------------------------------------------
    # Load best heads before saving final model
    # ------------------------------------------------------------
    sst_classifier.load_state_dict(torch.load(BEST_SST_HEAD_PATH, map_location=device))
    mnli_classifier.load_state_dict(torch.load(BEST_MNLI_HEAD_PATH, map_location=device))

    sst_classifier.eval()
    mnli_classifier.eval()
    merged_backbone.eval()

    # ------------------------------------------------------------
    # Save merged frozen backbone
    # ------------------------------------------------------------
    merged_backbone.save_pretrained(FINAL_MODEL_RESULTS_DIR / "merged_backbone")

    # ------------------------------------------------------------
    # Save tokenizer
    # Always google-bert/bert-base-uncased tokenizer
    # ------------------------------------------------------------
    tokenizer.save_pretrained(FINAL_MODEL_RESULTS_DIR / "tokenizer")

    # ------------------------------------------------------------
    # Save task-specific classifier heads
    # ------------------------------------------------------------
    torch.save(
        sst_classifier.state_dict(),
        FINAL_MODEL_RESULTS_DIR / "sst_classifier.pt"
    )

    torch.save(
        mnli_classifier.state_dict(),
        FINAL_MODEL_RESULTS_DIR / "mnli_classifier.pt"
    )

    # ------------------------------------------------------------
    # Save complete checkpoint
    # ------------------------------------------------------------
    torch.save(
        {
            "baseline": "weight_averaging",
            "tokenizer_id": TOKENIZER_ID,

            "sst_expert_dir": str(SST_EXPERT_DIR),
            "mnli_expert_dir": str(MNLI_EXPERT_DIR),
            "sst_backbone_dir": str(SST_BACKBONE_DIR),
            "mnli_backbone_dir": str(MNLI_BACKBONE_DIR),
            "merged_backbone_dir": str(MERGED_BACKBONE_DIR),

            "merged_backbone_state_dict": merged_backbone.state_dict(),
            "sst_classifier_state_dict": sst_classifier.state_dict(),
            "mnli_classifier_state_dict": mnli_classifier.state_dict(),

            "alpha": ALPHA,
            "beta": BETA,

            "sst_num_classes": 2,
            "mnli_num_classes": 3,

            "max_len": MAX_LEN,
            "batch_size": BATCH_SIZE,
            "best_epoch": best_epoch,
            "best_val_f1_avg": best_val_f1_avg,
        },
        FINAL_MODEL_RESULTS_DIR / "full_weight_averaging_checkpoint.pt"
    )

    # ------------------------------------------------------------
    # Save model metadata
    # ------------------------------------------------------------
    model_metadata = {
        "model_type": "weight_averaging_baseline",
        "backbone": "merged local SST + MNLI BERT backbones",
        "tokenizer_id": TOKENIZER_ID,

        "sst_expert_dir": str(SST_EXPERT_DIR),
        "mnli_expert_dir": str(MNLI_EXPERT_DIR),
        "sst_backbone_dir": str(SST_BACKBONE_DIR),
        "mnli_backbone_dir": str(MNLI_BACKBONE_DIR),
        "merged_backbone_dir": str(MERGED_BACKBONE_DIR),

        "alpha": float(ALPHA),
        "beta": float(BETA),

        "sst_num_classes": 2,
        "mnli_num_classes": 3,

        "max_len": int(MAX_LEN),
        "batch_size": int(BATCH_SIZE),
        "num_epochs_requested": int(NUM_EPOCHS),
        "num_epochs_completed": int(len(training_history)),

        "learning_rate": float(LEARNING_RATE),
        "weight_decay": float(WEIGHT_DECAY),

        "sst_head_params": int(sst_head_params),
        "mnli_head_params": int(mnli_head_params),
        "trainable_params": int(trainable_params),
        "frozen_backbone_params": int(frozen_backbone_params),

        "best_epoch": int(best_epoch),
        "best_val_f1_avg": float(best_val_f1_avg),

        "backbone_caching": True,
        "backbone_frozen": True,
    }

    with open(FINAL_MODEL_RESULTS_DIR / "model_metadata.json", "w") as f:
        json.dump(model_metadata, f, indent=4)

    print("Final weight averaging model saved in Results.")
    print(f"Path: {FINAL_MODEL_RESULTS_DIR}")
    print(f"Merged backbone: {FINAL_MODEL_RESULTS_DIR / 'merged_backbone'}")
    print(f"Tokenizer: {FINAL_MODEL_RESULTS_DIR / 'tokenizer'}")
    print(f"SST classifier: {FINAL_MODEL_RESULTS_DIR / 'sst_classifier.pt'}")
    print(f"MNLI classifier: {FINAL_MODEL_RESULTS_DIR / 'mnli_classifier.pt'}")
    print(f"Full checkpoint: {FINAL_MODEL_RESULTS_DIR / 'full_weight_averaging_checkpoint.pt'}")


In [ ]:
def input_fingerprint(input_ids, attention_mask, token_type_ids):
    """Same unpadded token identity used by the existing routed JSONL exports."""
    import hashlib
    import json
    tokens = [(int(i), int(t)) for i, t, m in
              zip(input_ids, token_type_ids, attention_mask) if m]
    return hashlib.sha256(json.dumps(tokens, separators=(",", ":")).encode("utf-8")).hexdigest()


def benchmark_inference(forward_batch, dataloader, device, *, warmup_batches=10, repetitions=3):
    """Time forward only, with CPU dispatch and synchronized GPU completion."""
    import time
    import numpy as np
    import torch
    from transformers import __version__ as transformers_version
    if warmup_batches < 1 or repetitions < 2 or len(dataloader.dataset) < 1:
        raise ValueError("Latency requires nonempty data, warmup >= 1 and repetitions >= 2")
    if not isinstance(dataloader.sampler, torch.utils.data.SequentialSampler):
        raise ValueError("Latency evaluation requires the same sequential inputs for all models")
    device = torch.device(device)

    def synchronize():
        if device.type == "cuda":
            torch.cuda.synchronize(device)

    batch_times, batch_sizes, per_repeat_ms = [], [], []
    lengths = set()
    dtype = None
    with torch.inference_mode():
        warmup_iter = iter(dataloader)
        for _ in range(warmup_batches):
            try:
                batch = next(warmup_iter)
            except StopIteration:
                warmup_iter = iter(dataloader)
                batch = next(warmup_iter)
            inputs = tuple(x.to(device) for x in batch[:3])
            warmup_output = forward_batch(inputs)
            dtype = str(warmup_output.dtype)
            del warmup_output
        synchronize()
        for _ in range(repetitions):
            repeat_ms, repeat_n = 0.0, 0
            for batch in dataloader:
                # Transfers, tokenization, dataloader work and metrics are not timed.
                inputs = tuple(x.to(device) for x in batch[:3])
                synchronize()
                start = time.perf_counter()
                logits = forward_batch(inputs)
                synchronize()
                elapsed_ms = 1000.0 * (time.perf_counter() - start)
                del logits
                n = int(inputs[0].shape[0])
                lengths.add(int(inputs[0].shape[1]))
                batch_times.append(elapsed_ms)
                batch_sizes.append(n)
                repeat_ms += elapsed_ms
                repeat_n += n
            if repeat_n != len(dataloader.dataset):
                raise ValueError("Latency pass did not cover the complete evaluation split")
            per_repeat_ms.append(repeat_ms / repeat_n)
    return {
        "scope": "forward on pretokenized device-resident inputs, including classifier and router if present",
        "excluded": "tokenization, data loading, host-device transfers, loss, metrics, export, FLOPs instrumentation",
        "timer": "perf_counter with CUDA synchronization before and after each forward",
        "warmup_batches": warmup_batches, "repetitions": repetitions,
        "num_examples_per_repeat": len(dataloader.dataset),
        "batch_size": dataloader.batch_size, "observed_batch_sizes": sorted(set(batch_sizes)),
        "observed_sequence_lengths": sorted(lengths),
        "mean_ms_per_batch": float(np.mean(batch_times)),
        "std_ms_per_batch": float(np.std(batch_times, ddof=1)),
        "mean_ms_per_example": float(sum(batch_times) / sum(batch_sizes)),
        "std_ms_per_example_across_repetitions": float(np.std(per_repeat_ms, ddof=1)),
        "ms_per_example_by_repeat": per_repeat_ms,
        "batch_ms_by_repeat": [batch_times[i * len(dataloader):(i + 1) * len(dataloader)]
                              for i in range(repetitions)],
        "device": str(device),
        "gpu": torch.cuda.get_device_name(device) if device.type == "cuda" else None,
        "torch_version": str(torch.__version__), "transformers_version": transformers_version,
        "dtype": dtype,
        "std_note": "Timing variability; distinct from variation between training seeds",
    }



In [ ]:

# Complete forward evaluation on the original sequential test loaders.
# Timing never uses cached features or FLOP instrumentation.
import csv
import hashlib
import json
import torch
from tqdm import tqdm
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report)

TEST_RESULTS_PATH = RESULTS_DIR / "weight_averaging_test_results.json"
if RUN_TRAINING:

    sst_classifier.load_state_dict(torch.load(BEST_SST_HEAD_PATH, map_location=device, weights_only=True))
    mnli_classifier.load_state_dict(torch.load(BEST_MNLI_HEAD_PATH, map_location=device, weights_only=True))
merged_backbone.eval()
sst_classifier.eval()
mnli_classifier.eval()


def compute_test_metrics(y_true, y_pred, task):
    average = "binary" if task == "sst" else "macro"
    labels = [0, 1] if task == "sst" else [0, 1, 2]
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, average=average, labels=labels, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, average=average, labels=labels, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, average=average, labels=labels, zero_division=0)),
        "f1_average": average,
        "confusion_matrix": confusion_matrix(y_true, y_pred, labels=labels).tolist(),
        "classification_report": classification_report(y_true, y_pred, labels=labels,
                                                       output_dict=True, zero_division=0),
    }


def evaluate_test_task(task, dataloader):
    if not isinstance(dataloader.sampler, torch.utils.data.SequentialSampler):
        raise ValueError("Prediction export requires a sequential test loader")
    head = sst_classifier if task == "sst" else mnli_classifier
    cost = new_inference_cost()
    y_true, y_pred = [], []
    total, loss_sum = 0, 0.0
    prediction_path = RESULTS_DIR / f"weight_averaging_{task}_predictions.csv"
    partial = prediction_path.with_suffix(".csv.partial")
    with partial.open("w", newline="", encoding="utf-8") as handle, torch.no_grad():
        writer = csv.DictWriter(handle, fieldnames=["example_id", "input_sha256", "label", "prediction"])
        writer.writeheader()
        for batch in tqdm(dataloader, desc=f"Evaluating {task} test"):
            input_ids, attention_mask, token_type_ids, labels = [x.to(device) for x in batch]
            with InferenceFlopCounter() as backbone_counter:
                output = merged_backbone(input_ids=input_ids, attention_mask=attention_mask,
                                         token_type_ids=token_type_ids)
            add_inference_cost(cost, backbone_counter, "backbone", input_ids)
            with InferenceFlopCounter() as head_counter:
                logits = head(output.pooler_output)
            add_inference_cost(cost, head_counter, "classifier")
            loss_sum += float(criterion(logits, labels).item()) * labels.size(0)
            predictions = logits.argmax(dim=-1).cpu().tolist()
            labels_cpu = labels.cpu().tolist()
            for ids, mask, types, label, prediction in zip(batch[0].tolist(), batch[1].tolist(),
                    batch[2].tolist(), labels_cpu, predictions):
                writer.writerow({"example_id": f"{task}:test:{total}",
                                 "input_sha256": input_fingerprint(ids, mask, types),
                                 "label": int(label), "prediction": int(prediction)})
                total += 1
            y_true.extend(labels_cpu)
            y_pred.extend(predictions)
    if total != len(dataloader.dataset) or not total:
        raise ValueError("Incomplete or empty test export")
    partial.replace(prediction_path)

    def forward_for_latency(inputs):
        output = merged_backbone(input_ids=inputs[0], attention_mask=inputs[1], token_type_ids=inputs[2])
        return head(output.pooler_output)

    return {
        "loss": loss_sum / total, "metrics": compute_test_metrics(y_true, y_pred, task),
        "num_samples": total, "inference_cost": finish_inference_cost(cost, total),
        "latency": benchmark_inference(forward_for_latency, dataloader, device),
        "predictions_csv": str(prediction_path),
        "predictions_sha256": hashlib.sha256(prediction_path.read_bytes()).hexdigest(),
    }


sst_test_results = evaluate_test_task("sst", sst_test_loader)
mnli_test_results = evaluate_test_task("mnli", mnli_test_loader)
test_avg_metrics = {
    key: (sst_test_results["metrics"][key] + mnli_test_results["metrics"][key]) / 2
    for key in ("accuracy", "precision", "recall", "f1")
}
test_results = {
    "baseline": "weight_averaging", "model_id": "google-bert/bert-base-uncased",
    "sst": sst_test_results, "mnli": mnli_test_results,
    "avg_between_tasks": test_avg_metrics,
    "metric_definitions": {"sst": "binary F1, positive label 1", "mnli": "macro F1, labels 0/1/2"},
    "max_len": int(MAX_LEN), "batch_size": int(BATCH_SIZE),
    "trainable_params": int(trainable_params),
    "sst_head_params": int(sst_head_params), "mnli_head_params": int(mnli_head_params),
    "best_epoch": int(best_epoch) if best_epoch is not None else None,
    "best_val_f1_avg": float(best_val_f1_avg) if best_val_f1_avg is not None else None,
    "best_validation_metric_note": "Historical checkpoint-selection value; existing checkpoints are not reselected",
    "training_executed_this_session": bool(RUN_TRAINING),
    "inference_backbone_caching": False,
    "training_cost": training_cost_report,
    "training_cost_note": "Historical saved training cost when evaluating; no additional training in evaluation mode",
    "evaluated_model_directory": str(FINAL_MODEL_RESULTS_DIR if RUN_TRAINING else SAVED_MODEL_DIR),
    "prediction_identity": "task:test:index plus SHA256 of unpadded input IDs and token type IDs",
}

test_results.update(tokenizer_id=TOKENIZER_ID,
                    alpha=float(ALPHA) if ALPHA is not None else None,
                    beta=float(BETA) if BETA is not None else None,
                    frozen_backbone_params=int(frozen_backbone_params),
                    backbone_frozen=True, backbone_caching=False,
                    merged_backbone_dir=str(MERGED_BACKBONE_DIR if RUN_TRAINING else SAVED_MODEL_DIR / "merged_backbone"),
                    sst_expert_dir=str(SST_EXPERT_DIR), mnli_expert_dir=str(MNLI_EXPERT_DIR),
                    sst_backbone_dir=str(SST_BACKBONE_DIR), mnli_backbone_dir=str(MNLI_BACKBONE_DIR))

with TEST_RESULTS_PATH.open("w", encoding="utf-8") as handle:
    json.dump(test_results, handle, indent=4, allow_nan=False)
for task in ("sst", "mnli"):
    result = test_results[task]
    metrics, timing = result["metrics"], result["latency"]
    print(f"{task.upper()} F1 ({metrics['f1_average']}): {metrics['f1']:.6f}; "
          f"accuracy: {metrics['accuracy']:.6f}; latency: {timing['mean_ms_per_example']:.3f} ms/example")
    print("Predictions:", result["predictions_csv"])
print("Results saved:", TEST_RESULTS_PATH)


In [ ]:
# ============================================================
# PRINT ALL TEST METRICS FROM JSON
# ============================================================

import json
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# Load test results
# ------------------------------------------------------------
TEST_RESULTS_PATH = RESULTS_DIR / "weight_averaging_test_results.json"

with open(TEST_RESULTS_PATH, "r") as f:
    test_results = json.load(f)

# ------------------------------------------------------------
# Utility functions
# ------------------------------------------------------------
def print_main_metrics(test_results):
    rows = []

    for task_key, task_name in [("sst", "SST"), ("mnli", "MNLI")]:
        metrics = test_results[task_key]["metrics"]

        rows.append({
            "Task": task_name,
            "Loss": test_results[task_key]["loss"],
            "Accuracy": metrics["accuracy"],
            "Precision": metrics["precision"],
            "Recall": metrics["recall"],
            "F1": metrics["f1"],
            "F1 average": metrics["f1_average"],
        })

    avg = test_results["avg_between_tasks"]

    rows.append({
        "Task": "AVG",
        "Loss": np.nan,
        "Accuracy": avg["accuracy"],
        "Precision": avg["precision"],
        "Recall": avg["recall"],
        "F1": avg["f1"],
        "F1 average": "mean of binary SST and macro MNLI",
    })

    df = pd.DataFrame(rows)

    print("\n" + "=" * 90)
    print("MAIN TEST METRICS")
    print("=" * 90)
    print(df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))


def print_confusion_matrix(task_key, task_name):
    cm = test_results[task_key]["metrics"]["confusion_matrix"]

    labels = [f"Class {i}" for i in range(len(cm))]

    df_cm = pd.DataFrame(
        cm,
        index=[f"True {label}" for label in labels],
        columns=[f"Pred {label}" for label in labels]
    )

    print("\n" + "=" * 90)
    print(f"{task_name} CONFUSION MATRIX")
    print("=" * 90)
    print(df_cm.to_string())


def print_classification_report(task_key, task_name):
    report = test_results[task_key]["metrics"]["classification_report"]

    rows = []

    for key, value in report.items():
        if isinstance(value, dict):
            rows.append({
                "Class / Avg": key,
                "Precision": value.get("precision", np.nan),
                "Recall": value.get("recall", np.nan),
                "F1": value.get("f1-score", np.nan),
                "Support": value.get("support", np.nan),
            })

    df_report = pd.DataFrame(rows)

    print("\n" + "=" * 90)
    print(f"{task_name} CLASSIFICATION REPORT")
    print("=" * 90)
    print(df_report.to_string(index=False, float_format=lambda x: f"{x:.4f}"))


def print_experiment_info(test_results):
    info_keys = [
        "baseline",
        "tokenizer_id",

        "alpha",
        "beta",

        "max_len",
        "batch_size",

        "trainable_params",
        "sst_head_params",
        "mnli_head_params",
        "frozen_backbone_params",

        "best_epoch",
        "best_val_f1_avg",

        "backbone_caching",
        "backbone_frozen",

        "sst_expert_dir",
        "mnli_expert_dir",
        "sst_backbone_dir",
        "mnli_backbone_dir",
        "merged_backbone_dir",
    ]

    rows = []

    for key in info_keys:
        if key in test_results:
            rows.append({
                "Parameter": key,
                "Value": test_results[key]
            })

    df_info = pd.DataFrame(rows)

    print("\n" + "=" * 90)
    print("EXPERIMENT INFO")
    print("=" * 90)
    print(df_info.to_string(index=False))


# ------------------------------------------------------------
# Computational cost: separate training and inference sections
# ------------------------------------------------------------
def format_flops(value):
    return f"{value:,}" if value is not None else "Incomplete"


def print_training_cost(test_results):
    print("\n" + "=" * 100)
    print("TRAINING COMPUTATIONAL COST (FLOPs)")
    print("=" * 100)
    cost = test_results.get("training_cost")
    if cost is None:
        print("Training cost not available in this report.")
        return
    print("Coverage:", cost["coverage"]["status"])
    rows = []
    for task in ("sst", "mnli"):
        task_cost = cost["by_task"][task]
        rows.append({
            "Task": task.upper(),
            "Training FLOPs": format_flops(task_cost["training_flops"]),
            "Validation FLOPs": format_flops(task_cost["validation_flops"]),
            "Total FLOPs": format_flops(task_cost["total_flops"]),
        })
    rows.append({
        "Task": "TOTAL",
        "Training FLOPs": format_flops(cost["training_flops"]),
        "Validation FLOPs": format_flops(cost["validation_flops"]),
        "Total FLOPs": format_flops(cost["total_flops"]),
    })
    print(pd.DataFrame(rows).to_string(index=False))

    components = []
    labels = {
        "backbone_cache_train": "Backbone cache - train (once)",
        "classifier_forward_and_loss": "Active head forward + loss",
        "backward": "Backward",
        "optimizer_step": "Optimizer updates",
        "backbone_cache_validation": "Backbone cache - validation (once)",
        "validation_forward_and_loss": "Validation head forward + loss",
    }
    for task in ("sst", "mnli"):
        for key, label in labels.items():
            component = cost["by_task"][task]["components"].get(key)
            if component is not None:
                components.append({"Task": task.upper(), "Component": label,
                                   "FLOPs": format_flops(component["total_flops"])})
    print("\nCost by component")
    print(pd.DataFrame(components).to_string(index=False))

    epochs = []
    for record in cost.get("per_epoch", []):
        for task in ("sst", "mnli"):
            values = record["by_task"][task]
            epochs.append({
                "Epoch": record["epoch"], "Task": task.upper(),
                "Training FLOPs": format_flops(values["training_flops"]),
                "Validation FLOPs": format_flops(values["validation_flops"]),
                "Total FLOPs": format_flops(values["total_flops"]),
            })
    if epochs:
        print("\nCost by epoch and task")
        print(pd.DataFrame(epochs).to_string(index=False))
    print("\nEpochs counted:", cost["epochs_completed"])
    print("Optimizer steps:", cost["optimizer_steps"])
    print("MAX_LEN:", cost["max_len"], "| Batch size:", cost["batch_size"])
    print("Run totals include each backbone cache once and all completed epochs, including those after the best epoch.")
    print("Epoch tables exclude the one-time caches and model preparation.")
    preparation = cost["preparation_cost"]
    print("\nModel preparation (one-time weight averaging)")
    print("ALPHA:", preparation["alpha"], "| BETA:", preparation["beta"])
    print("Weight averaging FLOPs:", format_flops(preparation["total_flops"]))
    print("Preparation + training + validation FLOPs:", format_flops(cost["total_with_preparation_flops"]))
    print("Preparation is shared by both tasks and is counted once; test inference is separate.")
    if cost["coverage"]["uncovered_operators"]:
        print("Uncovered operators:", ", ".join(cost["coverage"]["uncovered_operators"]))


def print_inference_cost(test_results):
    print("\n" + "=" * 100)
    print("INFERENCE COMPUTATIONAL COST (FLOPs)")
    print("=" * 100)
    rows = []
    for task in ("sst", "mnli"):
        cost = test_results[task].get("inference_cost")
        if cost is None:
            print(task.upper() + ": inference cost not available in this report.")
            continue
        per_example = cost["flops_per_example"]
        rows.append({
            "Task": task.upper(),
            "Examples": cost["num_examples"],
            "Total FLOPs": format_flops(cost["total_flops"]),
            "FLOPs per example": f"{per_example:,.2f}" if per_example is not None else "Incomplete",
            "Coverage": cost["coverage"]["status"],
        })
        if cost["coverage"]["uncovered_operators"]:
            print(task.upper() + " uncovered operators:", ", ".join(cost["coverage"]["uncovered_operators"]))
    if rows:
        print(pd.DataFrame(rows).to_string(index=False))
        print("Backbone + active task head, inputs to logits; test loss and metrics excluded.")



# ------------------------------------------------------------
# Print everything
# ------------------------------------------------------------
print_main_metrics(test_results)

print_confusion_matrix("sst", "SST")
print_classification_report("sst", "SST")

print_confusion_matrix("mnli", "MNLI")
print_classification_report("mnli", "MNLI")

print_experiment_info(test_results)

print_training_cost(test_results)
print_inference_cost(test_results)

print("\nMetrics loaded from:")
print(TEST_RESULTS_PATH)

print("\nINFERENCE LATENCY — full backbone + classifier, no feature caching")
print(pd.DataFrame([
    {"Task": task.upper(), "ms/example": test_results[task]["latency"]["mean_ms_per_example"],
     "SD across timing repetitions": test_results[task]["latency"]["std_ms_per_example_across_repetitions"],
     "ms/batch": test_results[task]["latency"]["mean_ms_per_batch"],
     "batch size": test_results[task]["latency"]["batch_size"],
     "GPU": test_results[task]["latency"]["gpu"],
     "predictions CSV": test_results[task]["predictions_csv"]}
    for task in ("sst", "mnli")
]).to_string(index=False))
